# CBIS-DDSM Complete Dataset: Comprehensive EDA

**Objective**: Extract ALL individual images from complete CBIS-DDSM dataset  
**Dataset**: Mass + Calcification cases (both train and test)  
**Output**: Clean metadata for maximum data utilization  
**Strategy**: Single-view approach - each image is an independent training sample

---

## Expected Dataset Size

**Mass Cases**:
- Training: ~1,000 images
- Test: ~300 images

**Calcification Cases**:
- Training: ~1,300 images
- Test: ~330 images

**Total Expected: ~3,000 individual images** (vs ~1,300 with paired approach)

---

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

!pip install -q pydicom
import pydicom

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print("✓ Libraries loaded")

In [ ]:
# Configuration
DATASET_ROOT = Path('/content/drive/MyDrive/CBIS-DDSM-data')
DICOM_DIR = DATASET_ROOT / 'CBIS-DDSM'
CSV_DIR = DATASET_ROOT / 'csv'
OUTPUT_DIR = DATASET_ROOT / 'eda_complete'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Dataset: {DATASET_ROOT}")
print(f"  DICOM: {DICOM_DIR}")
print(f"  CSV: {CSV_DIR}")
print(f"  Output: {OUTPUT_DIR}")

# Verify
assert DATASET_ROOT.exists(), f"Not found: {DATASET_ROOT}"
assert CSV_DIR.exists(), f"Not found: {CSV_DIR}"
print("\n✓ Paths verified")

---
## 2. Load Metadata

In [ ]:
# Load all CSV files
csv_files = {
    'mass_train': CSV_DIR / 'mass_case_description_train_set.csv',
    'mass_test': CSV_DIR / 'mass_case_description_test_set.csv',
    'calc_train': CSV_DIR / 'calc_case_description_train_set.csv',
    'calc_test': CSV_DIR / 'calc_case_description_test_set.csv'
}

metadata = {}
for key, path in csv_files.items():
    if path.exists():
        metadata[key] = pd.read_csv(path)
        print(f"✓ {key:15s}: {len(metadata[key]):4d} rows")
    else:
        print(f"✗ {key:15s}: NOT FOUND")

print(f"\nLoaded {len(metadata)} metadata files")

In [ ]:
# Quick inspection
print("Metadata Structure:")
print("=" * 70)

for name, df in metadata.items():
    print(f"\n{name.upper()}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns[:5])}... ({len(df.columns)} total)")

    if 'pathology' in df.columns:
        print(f"  Pathology: {df['pathology'].value_counts().to_dict()}")

    if 'image view' in df.columns:
        print(f"  Views: {df['image view'].value_counts().to_dict()}")

---
## 3. Extract All Individual Images

In [ ]:
def extract_all_images(df, abnormality_type, split):
    """
    Extract ALL individual images (no pairing requirement).

    Each row in metadata CSV becomes one training sample.

    Args:
        df: Metadata DataFrame
        abnormality_type: 'mass' or 'calc'
        split: 'train' or 'test'

    Returns:
        DataFrame with one row per image
    """
    images = []

    for idx, row in df.iterrows():
        # Unique image ID
        image_id = (
            f"{abnormality_type}_{split}_"
            f"{row['patient_id']}_"
            f"{row['left or right breast']}_"
            f"{row['image view']}"
        )

        # Extract DICOM case directory name
        case_dir = Path(row['cropped image file path']).parts[0]

        # Determine label
        pathology = str(row['pathology']).upper()
        if 'BENIGN' in pathology:
            label = 0
            label_name = 'benign'
        elif 'MALIGNANT' in pathology:
            label = 1
            label_name = 'malignant'
        else:
            label = -1
            label_name = 'unknown'

        images.append({
            # Identification
            'image_id': image_id,
            'case_dir': case_dir,
            'patient_id': row['patient_id'],

            # Image characteristics
            'breast_side': row['left or right breast'],
            'view': row['image view'],
            'abnormality_type': abnormality_type,

            # Clinical info
            'pathology': row['pathology'],
            'label': label,
            'label_name': label_name,

            # Dataset split
            'split': split,

            # Additional metadata (if available)
            'assessment': row.get('assessment', None),
            'subtlety': row.get('subtlety', None),
            'mass_shape': row.get('mass shape', None) if abnormality_type == 'mass' else None,
            'mass_margins': row.get('mass margins', None) if abnormality_type == 'mass' else None,
            'calc_type': row.get('calc type', None) if abnormality_type == 'calc' else None,
            'calc_distribution': row.get('calc distribution', None) if abnormality_type == 'calc' else None
        })

    return pd.DataFrame(images)


print("Extracting ALL individual images...")
print("=" * 70)

In [ ]:
# Extract from all datasets
all_datasets = []

for key, df in metadata.items():
    # Parse key: 'mass_train' -> abnormality='mass', split='train'
    parts = key.split('_')
    abnormality_type = parts[0]  # 'mass' or 'calc'
    split = parts[1]              # 'train' or 'test'

    print(f"\n{key.upper()}:")
    images_df = extract_all_images(df, abnormality_type, split)
    all_datasets.append(images_df)

    print(f"  Extracted: {len(images_df)} images")
    print(f"  Views: {dict(images_df['view'].value_counts())}")
    print(f"  Labels: {dict(images_df['label_name'].value_counts())}")

# Combine all
complete_dataset = pd.concat(all_datasets, ignore_index=True)

print(f"\n{'='*70}")
print(f"TOTAL IMAGES EXTRACTED: {len(complete_dataset):,}")
print(f"{'='*70}")

---
## 4. Dataset Statistics

In [ ]:
print("\nCOMPREHENSIVE DATASET STATISTICS")
print("=" * 70)

print(f"\nOverall:")
print(f"  Total images: {len(complete_dataset):,}")
print(f"  Unique patients: {complete_dataset['patient_id'].nunique():,}")
print(f"  Unique cases: {complete_dataset['case_dir'].nunique():,}")

print(f"\nBy Split:")
for split in ['train', 'test']:
    count = (complete_dataset['split'] == split).sum()
    pct = count / len(complete_dataset) * 100
    print(f"  {split.capitalize():5s}: {count:4d} ({pct:5.1f}%)")

print(f"\nBy Abnormality Type:")
for abn_type in ['mass', 'calc']:
    count = (complete_dataset['abnormality_type'] == abn_type).sum()
    pct = count / len(complete_dataset) * 100
    print(f"  {abn_type.capitalize():5s}: {count:4d} ({pct:5.1f}%)")

print(f"\nBy View:")
for view, count in complete_dataset['view'].value_counts().items():
    pct = count / len(complete_dataset) * 100
    print(f"  {view:3s}: {count:4d} ({pct:5.1f}%)")

print(f"\nBy Pathology:")
for label_name, count in complete_dataset['label_name'].value_counts().items():
    pct = count / len(complete_dataset) * 100
    print(f"  {label_name.capitalize():9s}: {count:4d} ({pct:5.1f}%)")

# Class balance
benign = (complete_dataset['label'] == 0).sum()
malignant = (complete_dataset['label'] == 1).sum()
if malignant > 0:
    ratio = benign / malignant
    print(f"\nClass Balance: {ratio:.2f}:1 (benign:malignant)")

In [ ]:
# Detailed breakdown by category
print("\nDETAILED BREAKDOWN")
print("=" * 70)

breakdown = complete_dataset.groupby(
    ['split', 'abnormality_type', 'label_name']
).size().reset_index(name='count')

print("\n" + breakdown.to_string(index=False))

# Summary table
print("\nSUMMARY TABLE: Split × Type × Pathology")
print("-" * 70)
pivot = complete_dataset.pivot_table(
    index=['split', 'abnormality_type'],
    columns='label_name',
    values='image_id',
    aggfunc='count',
    fill_value=0,
    margins=True,
    margins_name='TOTAL'
)
print(pivot)

---
## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Split distribution
complete_dataset['split'].value_counts().plot(
    kind='bar', ax=axes[0, 0], color=['steelblue', 'coral']
)
axes[0, 0].set_title('Train vs Test Split', fontweight='bold', fontsize=12)
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=0)
for container in axes[0, 0].containers:
    axes[0, 0].bar_label(container)

# 2. Abnormality type
complete_dataset['abnormality_type'].value_counts().plot(
    kind='bar', ax=axes[0, 1], color=['mediumseagreen', 'tomato']
)
axes[0, 1].set_title('Mass vs Calcification', fontweight='bold', fontsize=12)
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_xlabel('')
axes[0, 1].tick_params(axis='x', rotation=0)
for container in axes[0, 1].containers:
    axes[0, 1].bar_label(container)

# 3. View distribution
complete_dataset['view'].value_counts().plot(
    kind='bar', ax=axes[0, 2], color=['purple', 'orange', 'green']
)
axes[0, 2].set_title('View Distribution', fontweight='bold', fontsize=12)
axes[0, 2].set_ylabel('Count')
axes[0, 2].set_xlabel('')
axes[0, 2].tick_params(axis='x', rotation=45)
for container in axes[0, 2].containers:
    axes[0, 2].bar_label(container)

# 4. Pathology distribution
complete_dataset['label_name'].value_counts().plot(
    kind='bar', ax=axes[1, 0], color=['dodgerblue', 'crimson']
)
axes[1, 0].set_title('Benign vs Malignant', fontweight='bold', fontsize=12)
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_xlabel('')
axes[1, 0].tick_params(axis='x', rotation=0)
for container in axes[1, 0].containers:
    axes[1, 0].bar_label(container)

# 5. Train split breakdown
train_data = complete_dataset[complete_dataset['split'] == 'train']
train_breakdown = train_data.groupby(['abnormality_type', 'label_name']).size().unstack(fill_value=0)
train_breakdown.plot(kind='bar', ax=axes[1, 1], color=['dodgerblue', 'crimson'])
axes[1, 1].set_title('Training Set Breakdown', fontweight='bold', fontsize=12)
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_xlabel('')
axes[1, 1].legend(title='Pathology')
axes[1, 1].tick_params(axis='x', rotation=0)

# 6. Test split breakdown
test_data = complete_dataset[complete_dataset['split'] == 'test']
test_breakdown = test_data.groupby(['abnormality_type', 'label_name']).size().unstack(fill_value=0)
test_breakdown.plot(kind='bar', ax=axes[1, 2], color=['dodgerblue', 'crimson'])
axes[1, 2].set_title('Test Set Breakdown', fontweight='bold', fontsize=12)
axes[1, 2].set_ylabel('Count')
axes[1, 2].set_xlabel('')
axes[1, 2].legend(title='Pathology')
axes[1, 2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dataset_overview.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualizations saved")

---
## 6. DICOM Verification

In [ ]:
def find_dicom_file(case_dir_name, base_path):
    """
    Find DICOM file in case directory (3-level nesting).
    Structure: case_dir / date / series / file.dcm
    """
    case_path = base_path / case_dir_name

    try:
        for date_folder in case_path.iterdir():
            if date_folder.is_dir():
                for series_folder in date_folder.iterdir():
                    if series_folder.is_dir():
                        for dcm_file in series_folder.iterdir():
                            if dcm_file.suffix == '.dcm':
                                return dcm_file
    except:
        pass
    return None


# Sample 10 cases to verify DICOM accessibility
print("Verifying DICOM accessibility...")
print("=" * 70)

sample_cases = complete_dataset['case_dir'].sample(min(10, len(complete_dataset))).tolist()
accessible = 0
failed = []

for case_dir in sample_cases:
    dcm_file = find_dicom_file(case_dir, DICOM_DIR)
    if dcm_file:
        accessible += 1
        # Read one for inspection
        if accessible == 1:
            dcm = pydicom.dcmread(str(dcm_file))
            img = dcm.pixel_array
            print(f"\nSample DICOM inspection:")
            print(f"  Case: {case_dir}")
            print(f"  Shape: {img.shape}")
            print(f"  Data type: {img.dtype}")
            print(f"  Value range: [{img.min()}, {img.max()}]")
            print(f"  File size: {dcm_file.stat().st_size / 1e6:.2f} MB")
    else:
        failed.append(case_dir)

print(f"\nAccessibility check:")
print(f"  Accessible: {accessible}/{len(sample_cases)}")
print(f"  Success rate: {accessible/len(sample_cases)*100:.1f}%")

if failed:
    print(f"\n  Failed cases: {failed}")

---
## 7. Create Train/Test Splits

In [ ]:
# Split into train and test
train_dataset = complete_dataset[complete_dataset['split'] == 'train'].copy()
test_dataset = complete_dataset[complete_dataset['split'] == 'test'].copy()

print("Dataset Splits Created:")
print("=" * 70)
print(f"\nTraining set:")
print(f"  Total: {len(train_dataset)} images")
print(f"  Mass: {(train_dataset['abnormality_type'] == 'mass').sum()}")
print(f"  Calc: {(train_dataset['abnormality_type'] == 'calc').sum()}")
print(f"  Benign: {(train_dataset['label'] == 0).sum()}")
print(f"  Malignant: {(train_dataset['label'] == 1).sum()}")

print(f"\nTest set:")
print(f"  Total: {len(test_dataset)} images")
print(f"  Mass: {(test_dataset['abnormality_type'] == 'mass').sum()}")
print(f"  Calc: {(test_dataset['abnormality_type'] == 'calc').sum()}")
print(f"  Benign: {(test_dataset['label'] == 0).sum()}")
print(f"  Malignant: {(test_dataset['label'] == 1).sum()}")

---
## 8. Save Datasets

In [ ]:
# Save complete dataset
complete_dataset.to_csv(OUTPUT_DIR / 'complete_dataset.csv', index=False)
print(f"✓ Saved: complete_dataset.csv ({len(complete_dataset)} images)")

# Save train/test splits
train_dataset.to_csv(OUTPUT_DIR / 'train_all_images.csv', index=False)
print(f"✓ Saved: train_all_images.csv ({len(train_dataset)} images)")

test_dataset.to_csv(OUTPUT_DIR / 'test_all_images.csv', index=False)
print(f"✓ Saved: test_all_images.csv ({len(test_dataset)} images)")

# Save mass-only splits (for comparison)
train_mass = train_dataset[train_dataset['abnormality_type'] == 'mass']
test_mass = test_dataset[test_dataset['abnormality_type'] == 'mass']

train_mass.to_csv(OUTPUT_DIR / 'train_mass_only.csv', index=False)
test_mass.to_csv(OUTPUT_DIR / 'test_mass_only.csv', index=False)
print(f"✓ Saved: mass-only splits ({len(train_mass)} train, {len(test_mass)} test)")

# Save calc-only splits
train_calc = train_dataset[train_dataset['abnormality_type'] == 'calc']
test_calc = test_dataset[test_dataset['abnormality_type'] == 'calc']

train_calc.to_csv(OUTPUT_DIR / 'train_calc_only.csv', index=False)
test_calc.to_csv(OUTPUT_DIR / 'test_calc_only.csv', index=False)
print(f"✓ Saved: calc-only splits ({len(train_calc)} train, {len(test_calc)} test)")

---
## 9. Summary Report

In [ ]:
# Generate summary JSON
summary = {
    'dataset': 'CBIS-DDSM Complete',
    'extraction_date': datetime.now().isoformat(),
    'total_images': int(len(complete_dataset)),
    'unique_patients': int(complete_dataset['patient_id'].nunique()),

    'splits': {
        'train': {
            'total': int(len(train_dataset)),
            'mass': int((train_dataset['abnormality_type'] == 'mass').sum()),
            'calc': int((train_dataset['abnormality_type'] == 'calc').sum()),
            'benign': int((train_dataset['label'] == 0).sum()),
            'malignant': int((train_dataset['label'] == 1).sum())
        },
        'test': {
            'total': int(len(test_dataset)),
            'mass': int((test_dataset['abnormality_type'] == 'mass').sum()),
            'calc': int((test_dataset['abnormality_type'] == 'calc').sum()),
            'benign': int((test_dataset['label'] == 0).sum()),
            'malignant': int((test_dataset['label'] == 1).sum())
        }
    },

    'class_distribution': {
        'benign': int((complete_dataset['label'] == 0).sum()),
        'malignant': int((complete_dataset['label'] == 1).sum()),
        'balance_ratio': float(benign / malignant) if malignant > 0 else 0.0
    },

    'view_distribution': {k: int(v) for k, v in complete_dataset['view'].value_counts().items()},
    'abnormality_distribution': {k: int(v) for k, v in complete_dataset['abnormality_type'].value_counts().items()},

    'outputs': {
        'complete_dataset': 'complete_dataset.csv',
        'train_all': 'train_all_images.csv',
        'test_all': 'test_all_images.csv',
        'train_mass': 'train_mass_only.csv',
        'test_mass': 'test_mass_only.csv',
        'train_calc': 'train_calc_only.csv',
        'test_calc': 'test_calc_only.csv',
        'visualization': 'dataset_overview.png'
    }
}

# Save summary
with open(OUTPUT_DIR / 'eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✓ Summary saved: eda_summary.json")

In [ ]:
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"""
DATASET EXTRACTION COMPLETE
{'-'*70}

Total Images Extracted:     {summary['total_images']:,}
Unique Patients:            {summary['unique_patients']:,}

TRAINING SET
{'-'*70}
Total:                      {summary['splits']['train']['total']:,}
  Mass cases:               {summary['splits']['train']['mass']:,}
  Calcification cases:      {summary['splits']['train']['calc']:,}
  Benign:                   {summary['splits']['train']['benign']:,}
  Malignant:                {summary['splits']['train']['malignant']:,}

TEST SET
{'-'*70}
Total:                      {summary['splits']['test']['total']:,}
  Mass cases:               {summary['splits']['test']['mass']:,}
  Calcification cases:      {summary['splits']['test']['calc']:,}
  Benign:                   {summary['splits']['test']['benign']:,}
  Malignant:                {summary['splits']['test']['malignant']:,}

CLASS BALANCE
{'-'*70}
Benign:Malignant ratio:     {summary['class_distribution']['balance_ratio']:.2f}:1

OUTPUT FILES
{'-'*70}
Location: {OUTPUT_DIR}

Main files:
  ✓ complete_dataset.csv        (all {summary['total_images']:,} images)
  ✓ train_all_images.csv        ({summary['splits']['train']['total']:,} images)
  ✓ test_all_images.csv         ({summary['splits']['test']['total']:,} images)

Subset files:
  ✓ train_mass_only.csv         ({summary['splits']['train']['mass']:,} images)
  ✓ test_mass_only.csv          ({summary['splits']['test']['mass']:,} images)
  ✓ train_calc_only.csv         ({summary['splits']['train']['calc']:,} images)
  ✓ test_calc_only.csv          ({summary['splits']['test']['calc']:,} images)

Documentation:
  ✓ eda_summary.json
  ✓ dataset_overview.png
""")

print("=" * 70)
print("\nNEXT STEPS:")
print("  1. Train single-view baseline on train_all_images.csv")
print(f"     Expected: {summary['splits']['train']['total']:,} training images")
print(f"     Target: 0.88-0.92 AUC (with this much data)")
print("\n  2. Compare mass-only vs mass+calc performance")
print("\n  3. Implement advanced methods (attention, RadImageNet, ensemble)")
print("=" * 70)
print("\n✓ EDA complete! Ready for training. 🎉")